# DuckDB GeoParquet Benchmarks

This notebook measures direct DuckDB queries against local STAC GeoParquet outputs.

The benchmark goal is to compare Parquet layouts, not hash generation speed. Run `scripts/sync-benchmark-data.sh` before timing queries. Use `scripts/generate-file-count-matched-geoparquet.sh` if you want to rebuild the 12-file hashed layout locally.

## Setup

Expected Python packages:

- `duckdb`
- `stac-hash`, only if you want to compute hash range parameters in Python

The notebook uses DuckDB's spatial extension for exact geometry queries and `httpfs` for the optional remote S3 benchmark.

In [49]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import statistics
import time

import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')
con.execute('INSTALL spatial')
con.execute('LOAD spatial')
con.execute('INSTALL httpfs')
con.execute('LOAD httpfs')
con.execute("""
CREATE OR REPLACE SECRET benchmark_public_s3 (
    TYPE s3,
    PROVIDER config,
    REGION 'us-west-2',
    ENDPOINT 's3.us-west-2.amazonaws.com',
    URL_STYLE 'path',
    USE_SSL true
)
""")
con.execute('PRAGMA threads = 8')
con.execute('SET enable_external_file_cache = false')

duckdb.__version__

'1.5.5'

## Dataset Variants

Point each local variant at a GeoParquet file or glob. Remote variants use the same Source Cooperative layouts over S3 so object-store listing, range reads, and network latency are included. Keep all variants semantically equivalent so row counts match across layouts.

In [50]:
DATASETS = {
    'microsoft': '../data/benchmarks/source/mspc-sentinel-2-l2a/*.parquet',
    'hashed_128_files': '../data/benchmarks/source/mspc-sentinel-2-l2a-sorted/**/*.parquet',
    'hashed_12_files': '../data/benchmarks/generated/mspc-sentinel-2-l2a-sorted-12-files/*.parquet',
}

SOURCE_COOPERATIVE_PREFIX = 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet'
REMOTE_DATASETS = {
    'remote_microsoft': f'{SOURCE_COOPERATIVE_PREFIX}/mspc-sentinel-2-l2a/*.parquet',
    'remote_hashed_128_files': f'{SOURCE_COOPERATIVE_PREFIX}/mspc-sentinel-2-l2a-sorted/**/*.parquet',
    'remote_hashed_12_files': f'{SOURCE_COOPERATIVE_PREFIX}/mspc-sentinel-2-l2a-sorted-12-files/*.parquet',
}

DATASETS, REMOTE_DATASETS

({'microsoft': '../data/benchmarks/source/mspc-sentinel-2-l2a/*.parquet',
  'hashed_128_files': '../data/benchmarks/source/mspc-sentinel-2-l2a-sorted/**/*.parquet',
  'hashed_12_files': '../data/benchmarks/generated/mspc-sentinel-2-l2a-sorted-12-files/*.parquet'},
 {'remote_microsoft': 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a/*.parquet',
  'remote_hashed_128_files': 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a-sorted/**/*.parquet',
  'remote_hashed_12_files': 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a-sorted-12-files/*.parquet'})

## Query Parameters

Use fixed AOIs, time windows, collections, and ids so runs are comparable. Choose IDs that exist in every dataset variant. The source data currently covers 2025 Sentinel-2 L2A items.

The following cell can auto-fill `PARAMS['id']` with a real item id if the placeholder is left unchanged.


In [51]:
PARAMS = {
    'collection': 'sentinel-2-l2a',
    'id': 'REPLACE_WITH_REAL_ITEM_ID',
    'start_datetime': datetime(2025, 6, 1, tzinfo=timezone.utc),
    'end_datetime': datetime(2025, 7, 1, tzinfo=timezone.utc),
    'minx': -109.0,
    'miny': 37.0,
    'maxx': -102.0,
    'maxy': 41.0,
    'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
    'max_cloud_cover': 20.0,
    # Fill these once the hash range strategy is selected.
    'min_hash': 0,
    'max_hash': 9223372036854775807,
}

PARAMS

{'collection': 'sentinel-2-l2a',
 'id': 'REPLACE_WITH_REAL_ITEM_ID',
 'start_datetime': datetime.datetime(2025, 6, 1, 0, 0, tzinfo=datetime.timezone.utc),
 'end_datetime': datetime.datetime(2025, 7, 1, 0, 0, tzinfo=datetime.timezone.utc),
 'minx': -109.0,
 'miny': 37.0,
 'maxx': -102.0,
 'maxy': 41.0,
 'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
 'max_cloud_cover': 20.0,
 'min_hash': 0,
 'max_hash': 9223372036854775807}

In [52]:
# Auto-fill a real item id for the needle-in-a-haystack query.
# Override PARAMS['id'] manually above if you want a specific item.
if PARAMS['id'] == 'REPLACE_WITH_REAL_ITEM_ID':
    PARAMS['id'] = con.execute(
        "SELECT id FROM read_parquet(?, hive_partitioning = false) LIMIT 1",
        [DATASETS['microsoft']],
    ).fetchone()[0]

PARAMS['id']


'S2B_MSIL2A_20250101T031029_R075_T52VCJ_20250101T050301'

## Helpers

In [53]:
@dataclass(frozen=True)
class BenchmarkResult:
    dataset: str
    query: str
    rows: int | None
    best_seconds: float
    median_seconds: float
    runs: tuple[float, ...]


def sql_literal(value):
    if isinstance(value, datetime):
        return "TIMESTAMPTZ '" + value.isoformat().replace('+00:00', 'Z') + "'"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    if value is None:
        return 'NULL'
    return str(value)


def render(template: str, dataset_glob: str, params: dict) -> str:
    values = {'parquet_glob': sql_literal(dataset_glob)}
    values.update({key: sql_literal(value) for key, value in params.items()})
    return template.format(**values)


def run_sql(sql: str):
    return con.execute(sql).fetchall()


def time_query(sql: str, repeats: int = 5) -> tuple[int | None, tuple[float, ...]]:
    rows = None
    timings = []
    for _ in range(repeats):
        started = time.perf_counter()
        result = con.execute(sql).fetchall()
        timings.append(time.perf_counter() - started)
        if len(result) == 1 and len(result[0]) == 1 and isinstance(result[0][0], int):
            rows = result[0][0]
        else:
            rows = len(result)
    return rows, tuple(timings)


def explain_analyze(sql: str) -> str:
    rows = con.execute('EXPLAIN ANALYZE ' + sql).fetchall()
    return '\n'.join(str(row[1] if len(row) > 1 else row[0]) for row in rows)


def explain_analyze_json(sql: str):
    rows = con.execute('EXPLAIN (ANALYZE, FORMAT json) ' + sql).fetchall()
    payload = rows[0][1] if len(rows[0]) > 1 else rows[0][0]
    return json.loads(payload)

## Parquet Metadata

Run this before timing queries. It verifies file counts, row groups, and whether the important columns have useful row-group min/max statistics.

In [54]:
metadata_sql = """
SELECT
    file_name,
    count(DISTINCT row_group_id) AS row_groups,
    max(row_group_num_rows) AS max_row_group_rows,
    sum(row_group_compressed_bytes) AS compressed_bytes
FROM parquet_metadata({parquet_glob})
GROUP BY file_name
ORDER BY file_name
"""

for name, glob in DATASETS.items():
    print('\n##', name)
    try:
        display(con.execute(render(metadata_sql, glob, PARAMS)).df())
    except Exception as error:
        print(error)


## microsoft


,file_name,row_groups,max_row_group_rows,compressed_bytes
0,../data/benchmarks/source/mspc-sentinel-2-l2a/...,161,2048,1.010473e+11
1,../data/benchmarks/source/mspc-sentinel-2-l2a/...,144,2048,9.619865e+10
2,../data/benchmarks/source/mspc-sentinel-2-l2a/...,211,2048,1.346860e+11
3,../data/benchmarks/source/mspc-sentinel-2-l2a/...,226,2048,1.577516e+11
4,../data/benchmarks/source/mspc-sentinel-2-l2a/...,234,2048,1.592138e+11
5,../data/benchmarks/source/mspc-sentinel-2-l2a/...,225,2048,1.498662e+11
6,../data/benchmarks/source/mspc-sentinel-2-l2a/...,232,2048,1.465726e+11
7,../data/benchmarks/source/mspc-sentinel-2-l2a/...,231,2048,1.465465e+11
8,../data/benchmarks/source/mspc-sentinel-2-l2a/...,225,2048,1.437244e+11
9,../data/benchmarks/source/mspc-sentinel-2-l2a/...,215,2048,1.363797e+11



## hashed_128_files


,file_name,row_groups,max_row_group_rows,compressed_bytes
0,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39307,5.473829e+09
1,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39307,5.607146e+09
2,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39308,5.866470e+09
3,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39307,5.287036e+09
4,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39308,5.906282e+09
...,...,...,...,...
123,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39308,5.430317e+09
124,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39307,5.536509e+09
125,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39308,5.454389e+09
126,../data/benchmarks/source/mspc-sentinel-2-l2a-...,1,39307,5.435148e+09



## hashed_12_files


,file_name,row_groups,max_row_group_rows,compressed_bytes
0,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,4.857495e+10
1,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,5.161138e+10
2,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,5.364427e+10
3,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,4.633041e+10
4,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,5.140086e+10
5,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,5.413710e+10
6,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,4.819585e+10
7,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,5.287735e+10
8,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,5.418228e+10
9,../data/benchmarks/generated/mspc-sentinel-2-l...,4,122880,4.735390e+10


In [55]:
stats_sql = """
SELECT
    path_in_schema,
    count(*) AS row_groups,
    min(stats_min_value) AS global_min,
    max(stats_max_value) AS global_max
FROM parquet_metadata({parquet_glob})
WHERE path_in_schema IN (
    'hash:hash',
    'datetime',
    'collection',
    'id',
    'bbox.xmin',
    'bbox.ymin',
    'bbox.xmax',
    'bbox.ymax'
)
GROUP BY path_in_schema
ORDER BY path_in_schema
"""

for name, glob in DATASETS.items():
    print('\n##', name)
    try:
        display(con.execute(render(stats_sql, glob, PARAMS)).df())
    except Exception as error:
        print(error)


## microsoft


,path_in_schema,row_groups,global_min,global_max
0,collection,2463,sentinel-2-l2a,sentinel-2-l2a
1,datetime,2463,2025-01-01 00:04:39.024+00,2025-12-31 23:51:41.025+00
2,id,2463,S2A_MSIL2A_20250101T004031_R002_T56TPT_2025010...,S2C_MSIL2A_20251231T235141_R073_T59UPV_2026010...



## hashed_128_files


,path_in_schema,row_groups,global_min,global_max
0,collection,128,sentinel-2-l2a,sentinel-2-l2a
1,datetime,128,2025-01-01 00:04:39.024+00,2025-12-31 23:51:41.025+00
2,hash:hash,128,1060626393919962612,974454625439932830
3,id,128,S2A_MSIL2A_20250101T004031_R002_T56TPT_2025010...,S2C_MSIL2A_20251231T235141_R073_T59UPV_2026010...



## hashed_12_files


,path_in_schema,row_groups,global_min,global_max
0,collection,48,sentinel-2-l2a,sentinel-2-l2a
1,datetime,48,2025-01-01 00:04:39.024+00,2025-12-31 23:51:41.025+00
2,hash:hash,48,1081327013277922969,9186779992691430940
3,id,48,S2A_MSIL2A_20250101T004031_R002_T56TPT_2025010...,S2C_MSIL2A_20251231T235141_R073_T59UPV_2026010...


## Query Suite

In [56]:
QUERIES = {
    'q01_full_dataset_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
""",
    'q02_time_range_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE datetime >= {start_datetime}
  AND datetime < {end_datetime}
""",
    'q03_bbox_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
""",
    'q04_stac_search_count': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
""",
    'q05_hash_range_search': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE "hash:hash" BETWEEN {min_hash} AND {max_hash}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
""",
    'q06_search_page_datetime_order': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
ORDER BY datetime, id
LIMIT 100
""",
    'q06_search_page_hash_order': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
ORDER BY "hash:hash", id
LIMIT 100
""",
    'q07_attribute_filter': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
  AND "eo:cloud_cover" <= {max_cloud_cover}
""",
    'q08_exact_geometry_intersects': """
SELECT count(*)
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE datetime >= {start_datetime}
  AND datetime < {end_datetime}
  AND bbox.xmax >= {minx}
  AND bbox.xmin <= {maxx}
  AND bbox.ymax >= {miny}
  AND bbox.ymin <= {maxy}
  AND ST_Intersects(geometry, ST_GeomFromText({aoi_wkt}))
""",
    'q09_grouped_aggregation': """
SELECT
    collection,
    date_trunc('month', datetime) AS month,
    count(*) AS items
FROM read_parquet({parquet_glob}, hive_partitioning = false)
GROUP BY collection, month
ORDER BY collection, month
""",
    'q10_collection_latest_items': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
ORDER BY datetime DESC
LIMIT 100
""",
    'q11_specific_id_lookup': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE id = {id}
""",
    'q11_scoped_id_lookup': """
SELECT id, collection, datetime
FROM read_parquet({parquet_glob}, hive_partitioning = false)
WHERE collection = {collection}
  AND id = {id}
""",
}

list(QUERIES)

['q01_full_dataset_count',
 'q02_time_range_count',
 'q03_bbox_count',
 'q04_stac_search_count',
 'q05_hash_range_search',
 'q06_search_page_datetime_order',
 'q06_search_page_hash_order',
 'q07_attribute_filter',
 'q08_exact_geometry_intersects',
 'q09_grouped_aggregation',
 'q10_collection_latest_items',
 'q11_specific_id_lookup',
 'q11_scoped_id_lookup']

## Run Benchmarks

This cell runs the full query suite across all dataset variants. Hash-only queries are automatically skipped for the Microsoft dataset because it does not have the `"hash:hash"` column.

Use `median_seconds` for comparisons. `best_seconds` is useful for spotting warm-cache potential, but it can be optimistic.


In [57]:
HASH_ONLY_QUERIES = {
    'q05_hash_range_search',
    'q06_search_page_hash_order',
}
HASHED_DATASET_SUFFIXES = ('hashed_128_files', 'hashed_12_files')
def dataset_has_hash(dataset_name: str) -> bool:
    return dataset_name.endswith(HASHED_DATASET_SUFFIXES)


def run_benchmark_matrix(datasets: dict[str, str], queries: dict[str, str], repeats: int = 5):
    results = []
    for dataset_name, dataset_glob in datasets.items():
        for query_name, template in queries.items():
            if query_name in HASH_ONLY_QUERIES and not dataset_has_hash(dataset_name):
                continue
            sql = render(template, dataset_glob, PARAMS)
            try:
                rows, timings = time_query(sql, repeats=repeats)
            except Exception as error:
                print(f'{dataset_name} / {query_name}: {error}')
                continue
            results.append(BenchmarkResult(
                dataset=dataset_name,
                query=query_name,
                rows=rows,
                best_seconds=min(timings),
                median_seconds=statistics.median(timings),
                runs=timings,
            ))
            print(f'{dataset_name} / {query_name}: rows={rows} best={min(timings):0.4f}s median={statistics.median(timings):0.4f}s')
    return results


def results_table(results: list[BenchmarkResult]):
    rows = [
        {
            'query': result.query,
            'dataset': result.dataset,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
        }
        for result in results
    ]
    if not rows:
        return pd.DataFrame(columns=['query', 'dataset', 'rows', 'best_seconds', 'median_seconds'])
    return pd.DataFrame(rows).sort_values(['query', 'median_seconds']).reset_index(drop=True)


results = run_benchmark_matrix(DATASETS, QUERIES, repeats=5)
summary = results_table(results)
display(summary)


microsoft / q01_full_dataset_count: rows=5031349 best=0.8521s median=0.8705s
microsoft / q02_time_range_count: rows=459093 best=0.1377s median=0.1416s
microsoft / q03_bbox_count: rows=8842 best=0.2135s median=0.2149s
microsoft / q04_stac_search_count: rows=783 best=0.1478s median=0.1499s
microsoft / q06_search_page_datetime_order: rows=100 best=0.1429s median=0.1449s
microsoft / q07_attribute_filter: rows=488 best=0.1490s median=0.1537s
microsoft / q08_exact_geometry_intersects: rows=783 best=0.1540s median=0.1859s
microsoft / q09_grouped_aggregation: rows=13 best=0.1961s median=0.1988s
microsoft / q10_collection_latest_items: rows=100 best=0.1869s median=0.1905s
microsoft / q11_specific_id_lookup: rows=1 best=0.1832s median=0.1839s
microsoft / q11_scoped_id_lookup: rows=1 best=0.1941s median=0.1957s
hashed_128_files / q01_full_dataset_count: rows=5031349 best=0.0320s median=0.0323s
hashed_128_files / q02_time_range_count: rows=459093 best=0.0147s median=0.0152s
hashed_128_files / q03_

,query,dataset,rows,best_seconds,median_seconds
0,q01_full_dataset_count,hashed_12_files,5031349,0.014133,0.014875
1,q01_full_dataset_count,hashed_128_files,5031349,0.031987,0.032251
2,q01_full_dataset_count,microsoft,5031349,0.852051,0.870463
3,q02_time_range_count,hashed_12_files,459093,0.004529,0.004574
4,q02_time_range_count,hashed_128_files,459093,0.014691,0.015225
5,q02_time_range_count,microsoft,459093,0.137697,0.141585
6,q03_bbox_count,hashed_12_files,8842,0.008538,0.009385
7,q03_bbox_count,hashed_128_files,8842,0.014915,0.015246
8,q03_bbox_count,microsoft,8842,0.213523,0.214881
9,q04_stac_search_count,hashed_12_files,783,0.005985,0.006046


### Remote Object-Store Run

Run this cell to benchmark the same query suite directly against Source Cooperative S3. These timings include object-store listing, network latency, and HTTP range reads, so compare them separately from the local-first results above.

In [58]:
remote_results = run_benchmark_matrix(REMOTE_DATASETS, QUERIES, repeats=3)
remote_summary = results_table(remote_results)
display(remote_summary)


remote_microsoft / q01_full_dataset_count: rows=5031349 best=2.5981s median=2.9397s
remote_microsoft / q02_time_range_count: rows=459093 best=3.6311s median=3.9620s
remote_microsoft / q03_bbox_count: rows=8842 best=71.0025s median=72.0031s
remote_microsoft / q04_stac_search_count: rows=783 best=12.0099s median=12.2135s
remote_microsoft / q06_search_page_datetime_order: rows=100 best=7.9093s median=8.7381s
remote_microsoft / q07_attribute_filter: rows=488 best=14.3093s median=14.6723s
remote_microsoft / q08_exact_geometry_intersects: rows=783 best=10.6404s median=11.3679s
remote_microsoft / q09_grouped_aggregation: rows=13 best=52.6225s median=54.3369s
remote_microsoft / q10_collection_latest_items: rows=100 best=26.3156s median=28.1502s
remote_microsoft / q11_specific_id_lookup: rows=1 best=25.9754s median=26.3912s
remote_microsoft / q11_scoped_id_lookup: rows=1 best=38.5690s median=38.6381s
remote_hashed_128_files / q01_full_dataset_count: rows=5031349 best=4.0914s median=4.3767s
remo

,query,dataset,rows,best_seconds,median_seconds
0,q01_full_dataset_count,remote_hashed_12_files,5031349,0.741087,0.797104
1,q01_full_dataset_count,remote_microsoft,5031349,2.598067,2.939739
2,q01_full_dataset_count,remote_hashed_128_files,5031349,4.091397,4.376748
3,q02_time_range_count,remote_hashed_12_files,459093,1.087656,1.094050
4,q02_time_range_count,remote_microsoft,459093,3.631058,3.962012
5,q02_time_range_count,remote_hashed_128_files,459093,4.874556,4.919652
6,q03_bbox_count,remote_hashed_12_files,8842,1.444669,1.483335
7,q03_bbox_count,remote_hashed_128_files,8842,5.284142,5.294268
8,q03_bbox_count,remote_microsoft,8842,71.002482,72.003055
9,q04_stac_search_count,remote_hashed_12_files,783,1.543881,1.648777


## Analyze Results

These derived tables make the first-pass interpretation easier. The speedup table compares median runtimes by query. The file summary helps identify whether differences are caused by hash ordering or by lower-level Parquet characteristics such as file size and row-group layout.


In [59]:
pivot = summary.pivot(index='query', columns='dataset', values='median_seconds')
speedups = pivot.copy()
if {'microsoft', 'hashed_12_files'}.issubset(speedups.columns):
    speedups['microsoft_vs_hashed_12_speedup'] = speedups['microsoft'] / speedups['hashed_12_files']
if {'hashed_128_files', 'hashed_12_files'}.issubset(speedups.columns):
    speedups['hashed_128_vs_hashed_12_speedup'] = speedups['hashed_128_files'] / speedups['hashed_12_files']

display(speedups.reset_index())

if 'remote_summary' in globals() and not remote_summary.empty:
    remote_pivot = remote_summary.pivot(index='query', columns='dataset', values='median_seconds')
    remote_speedups = remote_pivot.copy()
    if {'remote_microsoft', 'remote_hashed_12_files'}.issubset(remote_speedups.columns):
        remote_speedups['remote_microsoft_vs_hashed_12_speedup'] = remote_speedups['remote_microsoft'] / remote_speedups['remote_hashed_12_files']
    if {'remote_hashed_128_files', 'remote_hashed_12_files'}.issubset(remote_speedups.columns):
        remote_speedups['remote_hashed_128_vs_hashed_12_speedup'] = remote_speedups['remote_hashed_128_files'] / remote_speedups['remote_hashed_12_files']
    display(remote_speedups.reset_index())


dataset,query,hashed_128_files,hashed_12_files,microsoft,microsoft_vs_hashed_12_speedup,hashed_128_vs_hashed_12_speedup
0,q01_full_dataset_count,0.032251,0.014875,0.870463,58.517549,2.168101
1,q02_time_range_count,0.015225,0.004574,0.141585,30.952661,3.328333
2,q03_bbox_count,0.015246,0.009385,0.214881,22.895957,1.624508
3,q04_stac_search_count,0.014443,0.006046,0.149947,24.800210,2.388825
4,q05_hash_range_search,0.016729,0.006733,NaN,NaN,2.484600
5,q06_search_page_datetime_order,0.016459,0.009472,0.144857,15.292767,1.737593
6,q06_search_page_hash_order,0.015679,0.010341,NaN,NaN,1.516112
7,q07_attribute_filter,0.015809,0.007330,0.153744,20.975397,2.156838
8,q08_exact_geometry_intersects,0.015985,0.021185,0.185925,8.776355,0.754545
9,q09_grouped_aggregation,0.034884,0.014371,0.198808,13.833688,2.427357


dataset,query,remote_hashed_128_files,remote_hashed_12_files,remote_microsoft,remote_microsoft_vs_hashed_12_speedup,remote_hashed_128_vs_hashed_12_speedup
0,q01_full_dataset_count,4.376748,0.797104,2.939739,3.688025,5.490814
1,q02_time_range_count,4.919652,1.094050,3.962012,3.621417,4.496734
2,q03_bbox_count,5.294268,1.483335,72.003055,48.541317,3.569164
3,q04_stac_search_count,4.677643,1.648777,12.213530,7.407630,2.837038
4,q05_hash_range_search,4.490073,1.212228,NaN,NaN,3.703984
5,q06_search_page_datetime_order,4.545145,1.547722,8.738074,5.645763,2.936667
6,q06_search_page_hash_order,4.659922,1.644658,NaN,NaN,2.833369
7,q07_attribute_filter,4.365493,1.651420,14.672340,8.884681,2.643478
8,q08_exact_geometry_intersects,4.846160,1.598570,11.367901,7.111293,3.031559
9,q09_grouped_aggregation,7.999093,2.213163,54.336930,24.551702,3.614325


In [60]:
metadata_rows = []
for dataset, glob in DATASETS.items():
    df = con.execute(render(metadata_sql, glob, PARAMS)).df()
    metadata_rows.append({
        'dataset': dataset,
        'files': len(df),
        'row_groups': int(df['row_groups'].sum()),
        'compressed_gb': float(df['compressed_bytes'].sum() / 1_000_000_000),
        'median_row_groups_per_file': float(df['row_groups'].median()),
        'median_max_row_group_rows': float(df['max_row_group_rows'].median()),
    })

file_summary = pd.DataFrame(metadata_rows).sort_values('dataset').reset_index(drop=True)
display(file_summary)


,dataset,files,row_groups,compressed_gb,median_row_groups_per_file,median_max_row_group_rows
0,hashed_128_files,128,128,701.764738,1.0,39307.0
1,hashed_12_files,12,48,613.700940,4.0,122880.0
2,microsoft,12,2463,1598.982122,220.0,2048.0


### Initial Reading Checklist

- If `microsoft_vs_hashed_12_speedup` is high for `q04_stac_search_count`, the generated hashed 12-file layout is materially better for the STAC-style query.
- If `hashed_128_vs_hashed_12_speedup` is above 1, the 12-file rewrite is faster than the 128-file layout for that query. That is a file-count or rewrite effect, not a hash effect.
- If `q01_full_dataset_count` is dramatically different, do not attribute all wins to hash sorting. It usually indicates major Parquet-level differences such as compressed size, metadata, row groups, or schema encoding.
- Treat `q05_hash_range_search` as provisional until the hash bounds are computed from the actual query window instead of placeholder min/max values.


## How To Read The Summary

For each query, compare rows with the same `query` value:

- `microsoft` vs `hashed_12_files` isolates sort/layout effects while keeping file count at 12.
- `hashed_128_files` vs `hashed_12_files` shows how much file count changes timing for the same hashed data.
- `q01_full_dataset_count` is mostly a raw scan control. If this differs a lot, file count/compression/schema overhead may be dominating.
- `q04_stac_search_count` is the main STAC-style query to watch. A hashed win here is the strongest signal.
- `q05_hash_range_search` is only meaningful once `PARAMS['min_hash']` and `PARAMS['max_hash']` represent the AOI/time window. Until then, treat it as provisional.
- `q11_specific_id_lookup` is a control. Hash sorting is not expected to help much for a bare id lookup.

Use `median_seconds` for comparison.


## Inspect a Plan

Use this when a timing difference looks interesting. The key line to look for is `Total Files Read`; if that number drops, DuckDB is pruning files. Also inspect filters shown under `TABLE_SCAN`.

Start with `q04_stac_search_count` on `microsoft` and `hashed_12_files`, then compare their plans side by side.


In [61]:
dataset_name = 'hashed_12_files'
query_name = 'q04_stac_search_count'

sql = render(QUERIES[query_name], DATASETS[dataset_name], PARAMS)
print(sql)
print(explain_analyze(sql))


SELECT count(*)
FROM read_parquet('../data/benchmarks/generated/mspc-sentinel-2-l2a-sorted-12-files/*.parquet', hive_partitioning = false)
WHERE collection = 'sentinel-2-l2a'
  AND datetime >= TIMESTAMPTZ '2025-06-01T00:00:00Z'
  AND datetime < TIMESTAMPTZ '2025-07-01T00:00:00Z'
  AND bbox.xmax >= -109.0
  AND bbox.xmin <= -102.0
  AND bbox.ymax >= 37.0
  AND bbox.ymin <= 41.0

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT count(*) FROM read_parquet('../data/benchmarks/generated/mspc-sentinel-2-l2a-sorted-12-files/*.parquet', hive_partitioning = false) WHERE collection = 'sentinel-2-l2a'   AND datetime >= TIMESTAMPTZ '2025-06-01T00:00:00Z'   AND datetime < TIMESTAMPTZ '2025-07-01T00:00:00Z'   AND bbox.xmax >= -109.0   AND bbox.xmin <= -102.0   AND bbox.ymax >= 37.0   AND bbox.ymin <= 41.0 
┌───────────────────

In [62]:
# JSON profile for downstream parsing. This is mostly useful once you know which query/dataset pair matters.
# Uncomment when needed.
# profile = explain_analyze_json(sql)
# profile


## Export Results

Run this after `results` exists and you want to save the timings for sharing or comparison.


In [63]:
def write_results(results: list[BenchmarkResult], path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = [
        {
            'dataset': result.dataset,
            'query': result.query,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
            'runs': list(result.runs),
            'duckdb_version': duckdb.__version__,
        }
        for result in results
    ]
    path.write_text(json.dumps(rows, indent=2), encoding='utf-8')
    return path


write_results(results, '../benchmark-results/duckdb-geoparquet-results.json')
if 'remote_results' in globals():
    write_results(remote_results, '../benchmark-results/duckdb-geoparquet-remote-results.json')
